In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy
import squarify

In [4]:
df = pd.read_csv("data/heart.csv")
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


# Heart Disease Analysis

## 1. Dataset Selection

**1. Dataset name**
Heart Failure Prediction Dataset (heart.csv)

**2. Source**
Kaggle: https://www.kaggle.com/datasets/endofnight17j03/heart-failure-prediction-dataset?select=heart.csv

**3. Why I chose it**
- Heart failure is an interesting and important topic.
- I want to see which gender has higher cholesterol, and how age affects it.
- I go to the gym, so I want to see whether exercise-related measures link to heart disease.
- The dataset has 918 patients with a mix of numeric and categorical columns, so there is enough data for meaningful analysis and cleaning.

**4. Questions I am aiming to answer**
1. Which gender has more heart disease?
2. Which age group shows the most signs of heart disease?
3. Which gender has higher cholesterol, and does cholesterol change with age?
4. Do patients who get chest pain during exercise have more heart disease?
5. Do patients with a higher maximum heart rate have less heart disease?
6. Which type of chest pain is most linked to heart disease?
7. Does high resting blood pressure link to heart disease?
8. Which factors are most strongly related to heart disease overall?

## 2. Initial Data Exploration

In [6]:
df.shape # number of rows and coloums

(918, 12)

In [7]:
df.columns # coloumn names

Index(['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS',
       'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope',
       'HeartDisease'],
      dtype='str')

In [8]:
df.dtypes # data types

Age                 int64
Sex                   str
ChestPainType         str
RestingBP           int64
Cholesterol         int64
FastingBS           int64
RestingECG            str
MaxHR               int64
ExerciseAngina        str
Oldpeak           float64
ST_Slope              str
HeartDisease        int64
dtype: object

In [10]:
df.head() # first few rows

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [11]:
df.describe() # summary of table

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


In [12]:
pd.isnull(df).sum() #checking for missing values

Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64

In [13]:
df.duplicated().sum() # any duplciates

np.int64(0)

In [14]:
df.groupby("HeartDisease").size() # number of ppl with heart disease

HeartDisease
0    410
1    508
dtype: int64

In [17]:
df.groupby("Sex").size() #number of patients female or male

Sex
F    193
M    725
dtype: int64

In [18]:
df.groupby("ChestPainType").size() # how many ppl have chest pain

ChestPainType
ASY    496
ATA    173
NAP    203
TA      46
dtype: int64

### Exploration findings
- The dataset has 918 rows and 12 columns.
- There are 5 numeric columns plus `FastingBS` and `HeartDisease`, which are coded 0/1, and 5 text columns (`Sex`, `ChestPainType`, `RestingECG`, `ExerciseAngina`, `ST_Slope`).
- There are no missing (null) values and no duplicate rows.
- `HeartDisease` is fairly balanced: 508 patients with heart disease and 410 without.
- `Cholesterol` has a minimum of 0 and `RestingBP` has a minimum of 0. Neither is possible for a living patient, so these are hidden missing values to clean.
- `Oldpeak` has a minimum of -2.6, so some values are negative and need checking.
- The data is mostly male (725 males vs 193 females), which may affect gender comparisons.

## 3. Data Cleaning

In [21]:
df_before = df.copy() #keeps copy of data

In [23]:
df.duplicated().sum() #checking for duplicates

np.int64(0)

In [24]:
df.drop_duplicates(inplace=True) #removes dups
df.shape

(918, 12)

In [25]:
print("Cholesterol = 0:", (df["Cholesterol"] == 0).sum()) # counts missing vaules 
print("RestingBP = 0:", (df["RestingBP"] == 0).sum())
print("Oldpeak below 0:", (df["Oldpeak"] < 0).sum())

Cholesterol = 0: 172
RestingBP = 0: 1
Oldpeak below 0: 13


In [27]:
df[df["RestingBP"] == 0] # finding missing value

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
449,55,M,NAP,0,0,0,Normal,155,N,1.5,Flat,1


In [28]:
df = df[df["RestingBP"] > 0] #dropping the values
df = df.reset_index(drop=True)
df.shape

(917, 12)

In [29]:
df.loc[df["Cholesterol"] == 0, "Cholesterol"] = np.nan # turning zero into Nan
pd.isnull(df).sum()

Age                 0
Sex                 0
ChestPainType       0
RestingBP           0
Cholesterol       171
FastingBS           0
RestingECG          0
MaxHR               0
ExerciseAngina      0
Oldpeak             0
ST_Slope            0
HeartDisease        0
dtype: int64

In [30]:
df["CholesterolMissing"] = pd.isnull(df["Cholesterol"])
df["Cholesterol"] = df["Cholesterol"].fillna(df["Cholesterol"].median())
pd.isnull(df).sum()  #filling missing value with median

Age                   0
Sex                   0
ChestPainType         0
RestingBP             0
Cholesterol           0
FastingBS             0
RestingECG            0
MaxHR                 0
ExerciseAngina        0
Oldpeak               0
ST_Slope              0
HeartDisease          0
CholesterolMissing    0
dtype: int64

In [31]:
df["Cholesterol"] = df["Cholesterol"].astype(int) #changing the data type
df.dtypes

Age                     int64
Sex                       str
ChestPainType             str
RestingBP               int64
Cholesterol             int64
FastingBS               int64
RestingECG                str
MaxHR                   int64
ExerciseAngina            str
Oldpeak               float64
ST_Slope                  str
HeartDisease            int64
CholesterolMissing       bool
dtype: object